# GNN-Guided SA for PCB Placement

GPU/TPU-accelerated training for learned PCB component placement.
This notebook runs scaled experiments — more rollouts, more epochs, longer SA annealing — to address underfitting observed in local runs.

**Paper:** [adekoya2026_gat_pcb_placement.pdf](https://github.com/elcruzo/learned-pcb-placement)

**Authors:** [Ayomide Adekoya](https://github.com/elcruzo) · [Jeff Allo](https://github.com/jeff4444) · [Olu Afolabi](https://github.com/oluuafolabi)

> **Runtime:** Use GPU (T4/A100) or TPU v2. Go to Runtime → Change runtime type.

In [ ]:
!git clone https://github.com/elcruzo/learned-pcb-placement.git
%cd learned-pcb-placement
!pip install torch numpy matplotlib -q

# Uncomment for TPU support:
# !pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html -q

In [ ]:
import torch, sys
print(f'Python {sys.version}')
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
try:
    import torch_xla.core.xla_model as xm
    print(f'TPU available: {xm.xla_device()}')
except Exception:
    print('TPU: not available (GPU mode)')

from learn import device
print(f'\nSelected device: {device}')

In [ ]:
# Parse all PCB files and show stats
import os, sys
sys.path.insert(0, '.')
from placer import parse_kicad_pcb

for f in sorted(os.listdir('data')):
    if not f.endswith('.kicad_pcb'): continue
    board = parse_kicad_pcb(f'data/{f}')
    print(f'{f:35s}  components={len(board.components):4d}  nets={len(board.nets):4d}  board={board.width:.1f}x{board.height:.1f}mm')

In [ ]:
# Scaled experiments — 50 rollouts, 200 epochs (vs 10/80 local)
# This is where GPU/TPU pays off: 5x more training data, 2.5x more epochs

EPOCHS = 200
ROLLOUTS = 50

import json, time
import numpy as np
from learn import run_experiment
from placer import parse_kicad_pcb

data_dir = 'data'
os.makedirs('results', exist_ok=True)

t_global = time.time()
for f in sorted(os.listdir(data_dir)):
    if not f.endswith('.kicad_pcb'): continue
    name = f.replace('.kicad_pcb', '')
    result_path = f'results/{name}.json'
    board = parse_kicad_pcb(f'{data_dir}/{f}')
    if len(board.components) < 5:
        continue
    print(f'\n===== {name} ({len(board.components)} components) =====')
    results = run_experiment(board, name, epochs=EPOCHS, rollouts_n=ROLLOUTS)
    with open(result_path, 'w') as fp:
        json.dump(results, fp, indent=2, default=lambda x: float(x) if isinstance(x, np.floating) else x)
    print(f'saved {result_path}')

print(f'\n{"="*60}')
print(f'ALL EXPERIMENTS DONE in {time.time()-t_global:.1f}s')

In [ ]:
# Generate all plots
!python3 graphs.py

In [ ]:
# Display results summary
import json, os
print(f'{"Board":<25} {"Comp":>5} {"Nets":>5} {"SA Cost":>10} {"GNN Cost":>10} {"SA HPWL":>10} {"GNN HPWL":>10} {"HPWL Δ":>8}')
print('-' * 90)
for f in sorted(os.listdir('results')):
    if not f.endswith('.json') or f == 'baseline.json': continue
    with open(f'results/{f}') as fp:
        r = json.load(fp)
    if 'sa_gnn' not in r: continue
    delta = (r['sa_gnn']['hpwl'] - r['sa_baseline']['hpwl']) / r['sa_baseline']['hpwl'] * 100
    print(f'{r["pcb"]:<25} {r["components"]:>5} {r["nets"]:>5} {r["sa_baseline"]["cost"]:>10.1f} {r["sa_gnn"]["cost"]:>10.1f} {r["sa_baseline"]["hpwl"]:>10.1f} {r["sa_gnn"]["hpwl"]:>10.1f} {delta:>7.1f}%')

In [ ]:
# Display plots
from IPython.display import Image, display
import os
for f in sorted(os.listdir('results')):
    if f.endswith('.png'):
        print(f'\n--- {f} ---')
        display(Image(f'results/{f}'))

In [ ]:
# Save results to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

import shutil
dst = '/content/drive/MyDrive/learned-pcb-placement-results'
os.makedirs(dst, exist_ok=True)
for f in os.listdir('results'):
    shutil.copy2(f'results/{f}', f'{dst}/{f}')
print(f'Saved {len(os.listdir(dst))} files to {dst}')